In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import torch
import inverse_pinn_training


In [2]:
# Test one complete run first

test_result = inverse_pinn_training.train_one_run(
    run_number=1
)

test_result



Starting Run 1
Training observations: 2000
Epoch     1 | Total: 2.159467e+01 | r: 0.099905 | sigma: 0.399670


KeyboardInterrupt: 

In [ ]:
# 10 independent runs
#
# Same training logic as before.
# The only addition is that each trained model is retained so that
# the 10 price predictions can be averaged BEFORE computing the final MSE.

N_runs = 10
results = []
models = []

for run in range(1, N_runs + 1):

    result, model_run = inverse_pinn_training.train_one_run(
        run_number=run,
        return_model=True
    )

    results.append(
        result
    )

    models.append(
        model_run
    )

    pd.DataFrame(
        results
    ).to_csv(
        "european_10_run_results.csv",
        index=False
    )


In [ ]:
results_df = pd.read_csv(
    "european_10_run_results.csv"
)


In [ ]:
# ============================================================
# 10-run summary
#
# Parameters:
#   average the 10 estimated r and sigma values as before.
#
# Prediction error:
#   DO NOT average the 10 individual MSE values for the main result.
#   Instead:
#       1. evaluate all 10 trained PINNs on the SAME fixed test mesh,
#       2. average their price predictions pointwise,
#       3. compute ONE final MSE from that averaged prediction.
#
# The fixed test mesh is the same 100 x 100 mesh used inside
# inverse_pinn_training.train_one_run().
# ============================================================


# ------------------------------------------------------------
# 1. Parameter summary: same 10-run mean/std logic as before
# ------------------------------------------------------------

parameter_summary = results_df[
    [
        "r_estimate",
        "sigma_estimate",
        "r_error_percent",
        "sigma_error_percent"
    ]
].agg(
    ["mean", "std"]
)

print("10-run parameter summary:")
display(
    parameter_summary
)


# ------------------------------------------------------------
# 2. Build the SAME fixed 100 x 100 PDE test mesh
# ------------------------------------------------------------

N_test = 100

device = inverse_pinn_training.device
dtype = inverse_pinn_training.dtype

t_min = inverse_pinn_training.t_min
t_max = inverse_pinn_training.t_max

S_min = inverse_pinn_training.S_min
S_max = inverse_pinn_training.S_max

K = inverse_pinn_training.K
T = inverse_pinn_training.T

r_true = inverse_pinn_training.r_true
sigma_true = inverse_pinn_training.sigma_true


t_test_values = torch.linspace(
    t_min,
    t_max,
    N_test,
    dtype=dtype,
    device=device
)

S_test_values = torch.linspace(
    S_min,
    S_max,
    N_test,
    dtype=dtype,
    device=device
)

t_mesh, S_mesh = torch.meshgrid(
    t_test_values,
    S_test_values,
    indexing="ij"
)

t_test = t_mesh.reshape(
    -1,
    1
)

S_test = S_mesh.reshape(
    -1,
    1
)


# ------------------------------------------------------------
# 3. Analytical Black-Scholes true prices
# ------------------------------------------------------------

with torch.no_grad():

    V_true = inverse_pinn_training.black_scholes_call(
        t=t_test,
        S=S_test,
        K=K,
        T=T,
        r=r_true,
        sigma=sigma_true
    )


# ------------------------------------------------------------
# 4. Get the 10 PINN price predictions on the SAME test mesh
# ------------------------------------------------------------

V_pinn_runs = []

with torch.no_grad():

    for run_number, model_run in enumerate(
        models,
        start=1
    ):

        model_run.eval()

        # PINN predicts normalized u = V / K
        u_pinn_run = model_run(
            t_test,
            S_test
        )

        # Convert back to physical option price V
        V_pinn_run = (
            K
            * u_pinn_run
        )

        V_pinn_runs.append(
            V_pinn_run
        )


# Shape:
# [10 runs, 10000 test points, 1]
V_pinn_stack = torch.stack(
    V_pinn_runs,
    dim=0
)


# ------------------------------------------------------------
# 5. FIRST average the 10 price predictions pointwise
# ------------------------------------------------------------

V_pinn_mean = torch.mean(
    V_pinn_stack,
    dim=0
)


# ------------------------------------------------------------
# 6. THEN compute the final MSE
# ------------------------------------------------------------

pde_mse_from_average_prediction = torch.mean(
    (
        V_true
        - V_pinn_mean
    ) ** 2
).item()


# Correlation is also evaluated using the averaged prediction
V_true_np = (
    V_true
    .detach()
    .cpu()
    .numpy()
    .reshape(-1)
)

V_pinn_mean_np = (
    V_pinn_mean
    .detach()
    .cpu()
    .numpy()
    .reshape(-1)
)

pde_correlation_from_average_prediction = np.corrcoef(
    V_true_np,
    V_pinn_mean_np
)[
    0,
    1
]


# ------------------------------------------------------------
# 7. Average r and sigma across the same 10 runs
# ------------------------------------------------------------

r_mean = results_df[
    "r_estimate"
].mean()

sigma_mean = results_df[
    "sigma_estimate"
].mean()


# ------------------------------------------------------------
# 8. Final 10-run aggregate result
# ------------------------------------------------------------

final_10_run_result = pd.DataFrame(
    [
        {
            "N_runs":
                N_runs,

            "r_estimate_mean":
                r_mean,

            "r_estimate_std":
                results_df[
                    "r_estimate"
                ].std(),

            "sigma_estimate_mean":
                sigma_mean,

            "sigma_estimate_std":
                results_df[
                    "sigma_estimate"
                ].std(),

            "r_error_percent_mean":
                results_df[
                    "r_error_percent"
                ].mean(),

            "r_error_percent_std":
                results_df[
                    "r_error_percent"
                ].std(),

            "sigma_error_percent_mean":
                results_df[
                    "sigma_error_percent"
                ].mean(),

            "sigma_error_percent_std":
                results_df[
                    "sigma_error_percent"
                ].std(),

            # MAIN prediction metric:
            # average 10 predictions first, then compute MSE
            "pde_mse_average_prediction":
                pde_mse_from_average_prediction,

            "pde_correlation_average_prediction":
                pde_correlation_from_average_prediction
        }
    ]
)

final_10_run_result.to_csv(
    "european_10_run_average_prediction_summary.csv",
    index=False
)

print()
print("Final 10-run aggregate result:")
display(
    final_10_run_result
)


# ------------------------------------------------------------
# 9. Save the averaged test prediction itself
# ------------------------------------------------------------

average_prediction_df = pd.DataFrame(
    {
        "t":
            t_test
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "S":
            S_test
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "V_true":
            V_true_np,

        "V_pinn_average_10_runs":
            V_pinn_mean_np
    }
)

average_prediction_df.to_csv(
    "european_10_run_average_prediction_test.csv",
    index=False
)

print()
print(
    "PDE MSE from average prediction =",
    pde_mse_from_average_prediction
)

print(
    "PDE correlation from average prediction =",
    pde_correlation_from_average_prediction
)


In [ ]:
import random
import numpy as np
import torch
import importlib


importlib.reload(inverse_pinn_training)

ADAM_DIAG_SEED = 42

random.seed(ADAM_DIAG_SEED)
np.random.seed(ADAM_DIAG_SEED)
torch.manual_seed(ADAM_DIAG_SEED)

(
    adam_result,
    adam_model,
    adam_history,
    adam_update_history
) = inverse_pinn_training.train_one_run(
    run_number=1,
    print_every=5000,
    return_model=True,
    track_history=True,
    track_adam_updates=True
)

In [ ]:
adam_diagnostic_checkpoint = {

    "model_state_dict":
        adam_model.state_dict(),

    "history":
        adam_history,


    "adam_update_history":
        adam_update_history,


    "result":
        adam_result,

    "seed":
        42,

    "optimizer":
        "Adam",

    "learning_rate":
        inverse_pinn_training.learning_rate,

    "num_epochs":
        inverse_pinn_training.num_epochs,

    "N_f":
        inverse_pinn_training.N_f,

    "N_terminal":
        inverse_pinn_training.N_terminal,

    "N_boundary":
        inverse_pinn_training.N_boundary,


    # --------------------------------------------------
    # Domain / option settings
    # --------------------------------------------------

    "t_min":
        inverse_pinn_training.t_min,

    "t_max":
        inverse_pinn_training.t_max,

    "S_min":
        inverse_pinn_training.S_min,

    "S_max":
        inverse_pinn_training.S_max,

    "K":
        inverse_pinn_training.K,

    "T":
        inverse_pinn_training.T,

    "r_true":
        inverse_pinn_training.r_true,

    "sigma_true":
        inverse_pinn_training.sigma_true,


    # --------------------------------------------------
    # Network architecture
    # --------------------------------------------------

    "hidden_width":
        inverse_pinn_training.hidden_width,

    "hidden_layers":
        inverse_pinn_training.hidden_layers
}


torch.save(
    adam_diagnostic_checkpoint,
    "european_inverse_adam_update_diagnostic_seed42.pt"
)

print(
    "Saved: european_inverse_adam_update_diagnostic_seed42.pt"
)

In [ ]:
import torch
import inverse_pinn_training
from inverse_pinn_training import (
    t_min, t_max, S_min, S_max, K, T,
    r_true, sigma_true, hidden_width, hidden_layers
)

N_runs = 10
checkpoint_paths = []

for run in range(1, N_runs + 1):
    print(f"Run {run}/{N_runs}")

    result, model_run, history_run = inverse_pinn_training.train_one_run(
        run_number=run,
        print_every=5000,
        return_model=True,
        track_history=True
    )

    checkpoint_run = {
        "history": history_run,
        "model_state_dict": model_run.state_dict(),
        "result": result,
        "t_min": t_min, "t_max": t_max,
        "S_min": S_min, "S_max": S_max,
        "K": K, "T": T,
        "r_true": r_true, "sigma_true": sigma_true,
        "hidden_width": hidden_width, "hidden_layers": hidden_layers
    }

    checkpoint_path = f"european_inverse_checkpoint_run{run}.pt"
    torch.save(checkpoint_run, checkpoint_path)
    checkpoint_paths.append(checkpoint_path)

    print(f"Saved: {checkpoint_path} | sigma={result['sigma_estimate']:.6f} | r={result['r_estimate']:.6f}")

print("\nAll 10 checkpoints saved:")
for p in checkpoint_paths:
    print(" ", p)